# 1. Train + quantize
Runtime > Change runtime type > **A100 GPU**. Put the whole `Banking_FInetuning` folder in your Google Drive root first (`MyDrive/Banking_FInetuning`). Everything is saved to Drive so a disconnect doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Banking_FInetuning
!nvidia-smi | head -15

In [ ]:
!pip install -q -r requirements-train.txt

## Data
Downloads Banking77 and writes `data/train|val|test.jsonl`.

In [ ]:
!python -m src.data.prepare && head -c 600 data/train.jsonl

## QLoRA fine-tune (about 10-20 min on A100)
Logged to MLflow in `./mlruns`.

In [ ]:
!python -m src.train.qlora

## Merge adapter into fp16 model

In [ ]:
!python -m src.quantize.merge

## AWQ int4
If `autoawq` fails to import (it pins old torch versions), tell Claude the error and we'll switch to llm-compressor.

In [ ]:
!python -m src.quantize.awq_quantize

## GGUF (Q4_K_M)
Builds llama.cpp in `/content` (CPU build is enough for conversion; ~3 min).

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j8 > /dev/null
!LLAMA_CPP=/content/llama.cpp bash scripts/make_gguf.sh

In [ ]:
!du -sh outputs/merged-fp16 outputs/merged-awq outputs/gguf/*

Model sizes above go into your README table. Next: notebook 2.